# 第12回: Flow Matching - フロー整合による行動生成

このノートブックでは、**Flow Matching** を使ったロボット行動生成を学びます。
Diffusion Policy と同様に確率的生成モデルですが、より直線的な経路を学習することで、
学習の安定性と推論効率が向上します。

**学習目標:**
- Continuous Normalizing Flows (CNF) の概念を理解する
- Flow Matching の原理と Diffusion との違いを把握する
- Robomimic データを用いた行動生成を実装する

## 0. 環境セットアップ

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Noto Sans CJK JP'
plt.rcParams['axes.unicode_minus'] = False
import math
from pathlib import Path

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用デバイス: {device}")

## 1. Flow Matching の概要

### 1.1 Continuous Normalizing Flows (CNF)

**Continuous Normalizing Flows (CNF)** は、ノイズ分布 $p_0$ からデータ分布 $p_1$ への
連続的な変換を学習するフレームワークです。

時刻 $t \in [0, 1]$ におけるベクトル場 $v_\theta(x_t, t)$ を使い、
常微分方程式 (ODE) に従ってサンプルを輸送します:

$$\frac{dx_t}{dt} = v_\theta(x_t, t)$$

- $t = 0$: ノイズ分布（ガウス分布）
- $t = 1$: データ分布（行動データ）

### 1.2 Diffusion vs Flow Matching

| 特徴 | Diffusion | Flow Matching |
|------|-----------|---------------|
| **経路** | 確率的（SDE） | 決定的（ODE） |
| **ノイズ付加** | 段階的にノイズを加え/除去 | ノイズからデータへの直線的な経路を学習 |
| **学習目標** | ノイズ $\epsilon$ を予測 | ベクトル場 $v_\theta$ を予測 |
| **推論ステップ数** | 多い（50-1000） | 少ない（10-20で高品質） |
| **数学的複雑さ** | 高い（分散スケジュール等） | シンプル（線形補間） |

### 1.3 条件付きフローマッチング (Conditional Flow Matching)

Flow Matching の核心は、**条件付き確率パス**を使ったシンプルな学習です。

**線形補間パス:**

データ点 $x_1$ とノイズ $x_0 \sim \mathcal{N}(0, I)$ を結ぶ直線的な経路を考えます:

$$x_t = (1 - t) \cdot x_0 + t \cdot x_1$$

**ターゲットベクトル場:**

この経路に沿った速度場は:

$$u_t = x_1 - x_0$$

**損失関数:**

ニューラルネットワーク $v_\theta$ でこのベクトル場を近似します:

$$\mathcal{L} = \mathbb{E}_{t, x_0, x_1} \left[ \| v_\theta(x_t, t) - u_t \|^2 \right]$$

この損失を最小化することで、ノイズからデータへの最適な輸送を学習します。

### 1.4 Diffusion と比べた利点

1. **学習が安定**: 直線的な経路なので、複雑なノイズスケジュールが不要
2. **少ない推論ステップ**: Euler 積分 10-20 ステップで高品質な生成が可能
3. **数学的にシンプル**: 線形補間と MSE 損失のみ
4. **柔軟性**: 異なる確率パス（最適輸送など）に拡張可能

### 1.5 直感的な理解: 2D での Flow Matching 可視化

簡単な2D例で、Flow Matching がどのようにノイズからデータへサンプルを輸送するか見てみましょう。

In [ ]:
# 2D の簡単な例で Flow Matching の概念を可視化
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

np.random.seed(42)
n_samples = 200

# ノイズ点 (t=0) とデータ点 (t=1)
x0 = np.random.randn(n_samples, 2)  # ガウスノイズ
theta = np.random.uniform(0, 2*np.pi, n_samples)
r = 2.0 + 0.2 * np.random.randn(n_samples)
x1 = np.stack([r * np.cos(theta), r * np.sin(theta)], axis=1)  # 円形データ

for idx, t_val in enumerate([0.0, 0.33, 0.66, 1.0]):
    xt = (1 - t_val) * x0 + t_val * x1  # 線形補間
    axes[idx].scatter(xt[:, 0], xt[:, 1], s=5, alpha=0.6)
    axes[idx].set_title(f"t = {t_val:.2f}", fontsize=14)
    axes[idx].set_xlim(-4, 4)
    axes[idx].set_ylim(-4, 4)
    axes[idx].set_aspect("equal")
    axes[idx].grid(True, alpha=0.3)

fig.suptitle("Flow Matching: ノイズ分布からデータ分布への輸送", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 2. モデルの実装

### 2.1 共通モジュール

まず、時刻埋め込みと条件付き ResBlock を定義します。
これらは Diffusion Policy と同じ構造です。

In [ ]:
class SinusoidalPositionEmbedding(nn.Module):
    """正弦波による時刻埋め込み。"""
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        half = self.dim // 2
        emb = math.log(10000) / (half - 1)
        emb = torch.exp(torch.arange(half, device=t.device, dtype=torch.float32) * -emb)
        emb = t[:, None].float() * emb[None, :]
        return torch.cat([emb.sin(), emb.cos()], dim=-1)


class ConditionalResBlock1D(nn.Module):
    """条件付き 1D 残差ブロック。"""
    def __init__(self, in_ch, out_ch, cond_dim):
        super().__init__()
        self.blocks = nn.Sequential(
            nn.Conv1d(in_ch, out_ch, 3, padding=1),
            nn.GroupNorm(8, out_ch),
            nn.GELU(),
            nn.Conv1d(out_ch, out_ch, 3, padding=1),
            nn.GroupNorm(8, out_ch),
        )
        self.cond_proj = nn.Linear(cond_dim, out_ch)
        self.residual = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
        self.act = nn.GELU()

    def forward(self, x, cond):
        h = self.blocks(x)
        # 条件を各時間ステップにブロードキャスト
        h = h + self.cond_proj(cond)[:, :, None]
        return self.act(h + self.residual(x))

print("共通モジュールを定義しました")

### 2.2 MLP ベースの速度場ネットワーク

最もシンプルなアーキテクチャとして、全結合層を使った速度場ネットワークを実装します。

In [ ]:
class FlowMatchingVelocityNet(nn.Module):
    """MLP ベースの速度場ネットワーク。"""
    def __init__(self, action_dim, obs_dim, horizon, hidden_dim=256):
        super().__init__()
        self.time_embed = SinusoidalPositionEmbedding(64)

        input_dim = action_dim * horizon + obs_dim + 64  # action + obs + time
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, action_dim * horizon),
        )
        self.action_dim = action_dim
        self.horizon = horizon

    def forward(self, x, t, obs):
        """
        Args:
            x: (batch, horizon, action_dim) - ノイズ付き行動
            t: (batch,) - 時刻 [0, 1]
            obs: (batch, obs_dim) - 観測条件
        Returns:
            v: (batch, horizon, action_dim) - 予測速度場
        """
        t_emb = self.time_embed(t * 1000)  # 埋め込み用にスケール
        x_flat = x.reshape(x.shape[0], -1)
        h = torch.cat([x_flat, obs, t_emb], dim=-1)
        v = self.net(h)
        return v.reshape(x.shape[0], self.horizon, self.action_dim)

# 動作確認
net = FlowMatchingVelocityNet(action_dim=7, obs_dim=20, horizon=16)
x = torch.randn(4, 16, 7)
t = torch.rand(4)
obs = torch.randn(4, 20)
v = net(x, t, obs)
print(f"入力: x={x.shape}, t={t.shape}, obs={obs.shape}")
print(f"出力: v={v.shape}")
print(f"パラメータ数: {sum(p.numel() for p in net.parameters()):,}")

### 2.3 1D U-Net ベースの速度場ネットワーク

より高性能なアーキテクチャとして、1D U-Net を使った速度場ネットワークを実装します。
スキップ接続により、長い行動系列の細部を保存できます。

In [ ]:
class FlowMatchingUNet(nn.Module):
    """1D U-Net ベースの速度場ネットワーク。"""
    def __init__(self, action_dim, obs_dim, horizon, time_dim=64):
        super().__init__()
        cond_dim = time_dim + obs_dim
        self.time_embed = SinusoidalPositionEmbedding(time_dim)
        self.obs_proj = nn.Linear(obs_dim, obs_dim)

        # ダウンサンプリング
        self.down1 = ConditionalResBlock1D(action_dim, 64, cond_dim)
        self.down2 = ConditionalResBlock1D(64, 128, cond_dim)
        # ボトルネック
        self.mid = ConditionalResBlock1D(128, 128, cond_dim)
        # アップサンプリング（スキップ接続でチャンネル数が倍）
        self.up2 = ConditionalResBlock1D(256, 64, cond_dim)   # 128+128
        self.up1 = ConditionalResBlock1D(128, 64, cond_dim)   # 64+64
        self.final = nn.Conv1d(64, action_dim, 1)

    def forward(self, x, t, obs):
        """
        Args:
            x: (batch, horizon, action_dim) - ノイズ付き行動
            t: (batch,) - 時刻 [0, 1]
            obs: (batch, obs_dim) - 観測条件
        Returns:
            v: (batch, horizon, action_dim) - 予測速度場
        """
        t_emb = self.time_embed(t * 1000)
        obs_emb = self.obs_proj(obs)
        cond = torch.cat([t_emb, obs_emb], dim=-1)

        # (batch, horizon, action_dim) -> (batch, action_dim, horizon)
        x = x.transpose(1, 2)

        # U-Net パス
        h1 = self.down1(x, cond)        # -> (batch, 64, horizon)
        h2 = self.down2(h1, cond)       # -> (batch, 128, horizon)
        h = self.mid(h2, cond)          # -> (batch, 128, horizon)
        h = self.up2(torch.cat([h, h2], dim=1), cond)   # -> (batch, 64, horizon)
        h = self.up1(torch.cat([h, h1], dim=1), cond)   # -> (batch, 64, horizon)
        out = self.final(h)             # -> (batch, action_dim, horizon)

        return out.transpose(1, 2)      # -> (batch, horizon, action_dim)

# 動作確認
unet = FlowMatchingUNet(action_dim=7, obs_dim=20, horizon=16)
v = unet(x, t, obs)
print(f"U-Net 出力: v={v.shape}")
print(f"パラメータ数: {sum(p.numel() for p in unet.parameters()):,}")

## 3. FlowMatchingTrainer

学習とサンプリングのロジックを一つのクラスにまとめます。

In [ ]:
class FlowMatchingTrainer:
    """Flow Matching の学習・サンプリングクラス。"""
    def __init__(self, model, optimizer, device="cpu"):
        self.model = model.to(device)
        self.optimizer = optimizer
        self.device = device

    def train_step(self, actions, obs):
        """
        1回の学習ステップ。

        Args:
            actions: (batch, horizon, action_dim) - 教師行動データ (= x_1)
            obs: (batch, obs_dim) - 観測条件
        Returns:
            loss: スカラー損失値
        """
        batch_size = actions.shape[0]

        # 1. 時刻 t をランダムにサンプル: t ~ U[0, 1]
        t = torch.rand(batch_size, device=self.device)

        # 2. ノイズをサンプル (ソース分布 x_0)
        x0 = torch.randn_like(actions)

        # 3. 線形補間: x_t = (1 - t) * x_0 + t * x_1
        t_expand = t[:, None, None]
        x_t = (1 - t_expand) * x0 + t_expand * actions

        # 4. ターゲット速度: u_t = x_1 - x_0
        target_velocity = actions - x0

        # 5. 速度場を予測
        pred_velocity = self.model(x_t, t, obs)

        # 6. MSE 損失
        loss = F.mse_loss(pred_velocity, target_velocity)

        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
        return loss.item()

    @torch.no_grad()
    def sample(self, obs, shape, num_steps=20):
        """
        Euler 積分によるサンプリング。

        Args:
            obs: (batch, obs_dim) - 観測条件
            shape: サンプルの形状 (batch, horizon, action_dim)
            num_steps: 積分ステップ数
        Returns:
            x: (batch, horizon, action_dim) - 生成された行動
        """
        x = torch.randn(shape, device=self.device)
        dt = 1.0 / num_steps

        for i in range(num_steps):
            t = torch.full((shape[0],), i * dt, device=self.device)
            v = self.model(x, t, obs)
            x = x + v * dt

        return x

print("FlowMatchingTrainer を定義しました")

## 4. データの準備と学習

### 4.1 Robomimic データの読み込み

LeRobot 形式の Robomimic データを読み込みます。
データが利用できない場合は、ダミーデータで学習を行います。

In [ ]:
def load_robomimic_data(data_dir="data/robomimic", horizon=16):
    """Robomimic データを LeRobot 形式で読み込む。"""
    parquet_dir = Path(data_dir)

    # parquet ファイルを探す
    parquet_files = sorted(parquet_dir.glob("**/*.parquet"))
    if parquet_files:
        try:
            import pandas as pd
            dfs = [pd.read_parquet(f) for f in parquet_files[:5]]
            df = pd.concat(dfs, ignore_index=True)

            # 行動列と観測列を取得
            action_cols = [c for c in df.columns if c.startswith("action")]
            obs_cols = [c for c in df.columns if c.startswith("observation")]

            if action_cols and obs_cols:
                actions_raw = df[action_cols].values.astype(np.float32)
                obs_raw = df[obs_cols].values.astype(np.float32)

                # ウィンドウに切り出し
                actions_list, obs_list = [], []
                for start in range(0, len(actions_raw) - horizon, horizon // 2):
                    actions_list.append(actions_raw[start:start+horizon])
                    obs_list.append(obs_raw[start])
                if actions_list:
                    print(f"Robomimic データを読み込みました: {len(actions_list)} サンプル")
                    return (torch.tensor(np.array(actions_list)),
                            torch.tensor(np.array(obs_list)))
        except Exception as e:
            print(f"データ読み込みエラー: {e}")

    return None

data = load_robomimic_data()

In [ ]:
# データが無い場合はダミーデータを生成
if data is None:
    print("Robomimic データが見つかりません。ダミーデータを生成します。")
    n_samples = 1000
    horizon = 16
    action_dim = 7
    obs_dim = 20

    # 複数のパターンを持つダミー行動データ
    actions_all = []
    obs_all = []
    for i in range(n_samples):
        pattern = i % 3
        t_seq = torch.linspace(0, 1, horizon).unsqueeze(-1)  # (horizon, 1)
        if pattern == 0:
            # 正弦波パターン
            action = torch.sin(2 * np.pi * t_seq * torch.randn(1, action_dim))
        elif pattern == 1:
            # 線形パターン
            action = t_seq * torch.randn(1, action_dim) * 0.5
        else:
            # 放物線パターン
            action = (t_seq ** 2) * torch.randn(1, action_dim)
        action += 0.05 * torch.randn(horizon, action_dim)  # ノイズ付加
        actions_all.append(action)
        obs_all.append(torch.randn(obs_dim) * 0.1 + pattern)

    actions_data = torch.stack(actions_all)  # (n, horizon, action_dim)
    obs_data = torch.stack(obs_all)          # (n, obs_dim)
else:
    actions_data, obs_data = data

print(f"行動データ: {actions_data.shape}")
print(f"観測データ: {obs_data.shape}")

action_dim = actions_data.shape[-1]
obs_dim = obs_data.shape[-1]
horizon = actions_data.shape[1]

### 4.2 学習ループ

In [ ]:
# モデルとトレーナーの初期化
model = FlowMatchingUNet(
    action_dim=action_dim,
    obs_dim=obs_dim,
    horizon=horizon,
    time_dim=64,
)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-6)
trainer = FlowMatchingTrainer(model, optimizer, device=device)

# データをデバイスに移動
actions_data = actions_data.to(device)
obs_data = obs_data.to(device)

# 学習
n_epochs = 50
batch_size = 64
losses = []
n_samples = len(actions_data)

print(f"学習開始: {n_epochs} エポック, バッチサイズ {batch_size}")
for epoch in range(n_epochs):
    epoch_losses = []
    perm = torch.randperm(n_samples)
    for i in range(0, n_samples - batch_size + 1, batch_size):
        idx = perm[i:i+batch_size]
        loss = trainer.train_step(actions_data[idx], obs_data[idx])
        epoch_losses.append(loss)
    avg_loss = np.mean(epoch_losses)
    losses.append(avg_loss)
    if (epoch + 1) % 10 == 0:
        print(f"  エポック {epoch+1:3d}/{n_epochs}: 損失 = {avg_loss:.6f}")

print("学習完了!")

## 5. 予測結果の可視化

### 5.1 損失曲線

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(losses, linewidth=2)
plt.xlabel("エポック", fontsize=12)
plt.ylabel("損失 (MSE)", fontsize=12)
plt.title("Flow Matching 学習損失の推移", fontsize=14)
plt.grid(True, alpha=0.3)
plt.yscale("log")
plt.tight_layout()
plt.show()

### 5.2 生成行動と教師データの比較

In [ ]:
# サンプリング: 複数の観測条件で行動を生成
n_vis = 6
vis_idx = torch.randperm(len(obs_data))[:n_vis]
obs_vis = obs_data[vis_idx]
actions_gt = actions_data[vis_idx]

# 異なるステップ数で生成
steps_list = [5, 10, 20]
fig, axes = plt.subplots(n_vis, len(steps_list) + 1, figsize=(4 * (len(steps_list) + 1), 3 * n_vis))

for row in range(n_vis):
    # 教師データ
    for d in range(min(3, action_dim)):
        axes[row, 0].plot(actions_gt[row, :, d].cpu().numpy(), label=f"dim {d}")
    axes[row, 0].set_title("教師データ", fontsize=10)
    axes[row, 0].legend(fontsize=7)
    axes[row, 0].grid(True, alpha=0.3)

    # 各ステップ数での生成
    for col, n_steps in enumerate(steps_list):
        gen = trainer.sample(
            obs_vis[row:row+1],
            shape=(1, horizon, action_dim),
            num_steps=n_steps,
        )
        for d in range(min(3, action_dim)):
            axes[row, col+1].plot(gen[0, :, d].cpu().numpy(), label=f"dim {d}")
        axes[row, col+1].set_title(f"生成 ({n_steps} steps)", fontsize=10)
        axes[row, col+1].legend(fontsize=7)
        axes[row, col+1].grid(True, alpha=0.3)

plt.suptitle("教師行動 vs 生成行動 (推論ステップ数の比較)", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

### 5.3 フロー場の可視化 (2D 射影)

In [ ]:
# 行動空間の最初の2次元にフロー場を可視化
fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))

# 代表的な観測を使用
obs_ref = obs_data[0:1]

t_values = [0.0, 0.25, 0.5, 0.75]
grid_n = 12

for ax_idx, t_val in enumerate(t_values):
    # グリッド点を作成
    xx = np.linspace(-3, 3, grid_n)
    yy = np.linspace(-3, 3, grid_n)
    XX, YY = np.meshgrid(xx, yy)

    # グリッド点でベクトル場を評価
    grid_points = np.stack([XX.ravel(), YY.ravel()], axis=1)
    n_grid = len(grid_points)

    # 残りの次元はゼロで埋める
    x_full = torch.zeros(n_grid, horizon, action_dim, device=device)
    x_full[:, 0, 0] = torch.tensor(grid_points[:, 0], dtype=torch.float32)
    x_full[:, 0, 1] = torch.tensor(grid_points[:, 1], dtype=torch.float32)

    t_tensor = torch.full((n_grid,), t_val, device=device)
    obs_rep = obs_ref.expand(n_grid, -1)

    with torch.no_grad():
        v = trainer.model(x_full, t_tensor, obs_rep)

    vx = v[:, 0, 0].cpu().numpy().reshape(grid_n, grid_n)
    vy = v[:, 0, 1].cpu().numpy().reshape(grid_n, grid_n)

    speed = np.sqrt(vx**2 + vy**2)
    axes[ax_idx].quiver(XX, YY, vx, vy, speed, cmap="coolwarm", alpha=0.8)
    axes[ax_idx].set_title(f"t = {t_val:.2f}", fontsize=13)
    axes[ax_idx].set_xlabel("action dim 0")
    axes[ax_idx].set_ylabel("action dim 1")
    axes[ax_idx].set_aspect("equal")
    axes[ax_idx].grid(True, alpha=0.3)

fig.suptitle("学習された速度場の2D射影 (時刻ごとの変化)", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### 5.4 サンプリング過程の可視化

Euler 積分の各ステップで、サンプルがどのようにノイズからデータ分布に移動するかを可視化します。

In [ ]:
@torch.no_grad()
def sample_with_trajectory(trainer, obs, shape, num_steps=20):
    """各ステップの軌跡を記録しながらサンプリング。"""
    x = torch.randn(shape, device=trainer.device)
    trajectory = [x.cpu().clone()]
    dt = 1.0 / num_steps

    for i in range(num_steps):
        t = torch.full((shape[0],), i * dt, device=trainer.device)
        v = trainer.model(x, t, obs)
        x = x + v * dt
        trajectory.append(x.cpu().clone())

    return trajectory

# サンプリング軌跡の取得
n_traj = 8
obs_traj = obs_data[:1].expand(n_traj, -1)
trajectory = sample_with_trajectory(
    trainer, obs_traj, (n_traj, horizon, action_dim), num_steps=20
)

# 最初のタイムステップの最初の2次元を可視化
fig, ax = plt.subplots(figsize=(8, 8))
colors = plt.cm.viridis(np.linspace(0, 1, n_traj))

for s in range(n_traj):
    xs = [traj[s, 0, 0].item() for traj in trajectory]
    ys = [traj[s, 0, 1].item() for traj in trajectory]
    ax.plot(xs, ys, "-o", color=colors[s], markersize=3, alpha=0.7, linewidth=1.5)
    ax.plot(xs[0], ys[0], "x", color=colors[s], markersize=10)  # 開始点
    ax.plot(xs[-1], ys[-1], "s", color=colors[s], markersize=8)  # 終了点

ax.set_xlabel("action dim 0", fontsize=12)
ax.set_ylabel("action dim 1", fontsize=12)
ax.set_title("サンプリング軌跡 (x=開始, □=終了)", fontsize=14)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 演習問題

### 演習1: Flow Matching の train_step を実装せよ

線形補間によるノイズ付き行動の生成と、ターゲットベクトル場の計算を実装してください。

以下の `my_train_step` を完成させてください。

In [ ]:
def my_train_step(model, optimizer, actions, obs, device):
    """
    Flow Matching の学習ステップ。

    Args:
        model: 速度場ネットワーク
        optimizer: オプティマイザ
        actions: (batch, horizon, action_dim) - 教師データ (x_1)
        obs: (batch, obs_dim) - 観測条件
        device: デバイス
    Returns:
        loss: スカラー損失値
    """
    batch_size = actions.shape[0]

    # TODO: ランダムな時刻 t を [0, 1] からサンプル
    t = None  # ここを実装

    # TODO: ノイズ x_0 をサンプル
    x0 = None  # ここを実装

    # TODO: 線形補間で x_t を計算: x_t = (1-t)*x_0 + t*x_1
    x_t = None  # ここを実装

    # TODO: ターゲット速度を計算: u_t = x_1 - x_0
    target_velocity = None  # ここを実装

    # TODO: モデルで速度を予測して MSE 損失を計算
    pred_velocity = None  # ここを実装
    loss = None  # ここを実装

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    return loss.item()

<details>
<summary><b>回答例を表示</b></summary>

```python
def my_train_step(model, optimizer, actions, obs, device):
    batch_size = actions.shape[0]

    # ランダムな時刻 t を [0, 1] からサンプル
    t = torch.rand(batch_size, device=device)

    # ノイズ x_0 をサンプル
    x0 = torch.randn_like(actions)

    # 線形補間で x_t を計算
    t_expand = t[:, None, None]
    x_t = (1 - t_expand) * x0 + t_expand * actions

    # ターゲット速度を計算
    target_velocity = actions - x0

    # モデルで速度を予測して MSE 損失を計算
    pred_velocity = model(x_t, t, obs)
    loss = F.mse_loss(pred_velocity, target_velocity)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    return loss.item()
```

</details>

### 演習2: Euler 積分によるサンプリングを実装せよ

ノイズから開始して、学習した速度場に従って Euler 法で ODE を解くサンプリング関数を実装してください。

以下の `my_sample` を完成させてください。

In [ ]:
@torch.no_grad()
def my_sample(model, obs, shape, num_steps=20, device="cpu"):
    """
    Euler 積分によるサンプリング。

    Args:
        model: 学習済み速度場ネットワーク
        obs: (batch, obs_dim) - 観測条件
        shape: 出力形状 (batch, horizon, action_dim)
        num_steps: 積分ステップ数
        device: デバイス
    Returns:
        x: (batch, horizon, action_dim) - 生成された行動
    """
    # TODO: ノイズからスタート
    x = None  # ここを実装

    # TODO: dt を計算
    dt = None  # ここを実装

    # TODO: Euler 積分ループ
    # for each step:
    #   1. 現在の時刻 t を計算
    #   2. モデルで速度 v を予測
    #   3. x を更新: x = x + v * dt

    return x

<details>
<summary><b>回答例を表示</b></summary>

```python
@torch.no_grad()
def my_sample(model, obs, shape, num_steps=20, device="cpu"):
    # ノイズからスタート
    x = torch.randn(shape, device=device)

    # dt を計算
    dt = 1.0 / num_steps

    # Euler 積分ループ
    for i in range(num_steps):
        t = torch.full((shape[0],), i * dt, device=device)
        v = model(x, t, obs)
        x = x + v * dt

    return x
```

</details>

### 演習3: Diffusion Policy と推論速度・品質を比較せよ

推論ステップ数を変えて、Flow Matching の生成品質と速度のトレードオフを調べてください。
さらに余裕があれば、同じデータで Diffusion Policy（DDPM）を学習し比較してください。

以下のコードを完成させ、ステップ数ごとの推論時間と品質を計測してください。

In [ ]:
import time

steps_to_test = [1, 2, 5, 10, 20, 50]
n_eval = 50
obs_eval = obs_data[:n_eval]
gt_eval = actions_data[:n_eval]

results = {}
for n_steps in steps_to_test:
    # TODO: 推論時間を計測
    pass

    # TODO: 生成品質 (MSE) を計算
    pass

    # results[n_steps] = {"time": elapsed, "mse": mse_val}

# TODO: 結果をプロットする
# x軸: ステップ数、y軸左: MSE、y軸右: 推論時間

<details>
<summary><b>回答例を表示</b></summary>

```python
import time

steps_to_test = [1, 2, 5, 10, 20, 50]
n_eval = 50
obs_eval = obs_data[:n_eval]
gt_eval = actions_data[:n_eval]

results = {}
for n_steps in steps_to_test:
    # 推論時間を計測
    start = time.time()
    gen = trainer.sample(obs_eval, (n_eval, horizon, action_dim), num_steps=n_steps)
    elapsed = time.time() - start

    # 生成品質 (MSE) を計算
    mse_val = F.mse_loss(gen, gt_eval).item()
    results[n_steps] = {"time": elapsed, "mse": mse_val}
    print(f"  steps={n_steps:3d}: MSE={mse_val:.6f}, time={elapsed:.4f}s")

# 結果をプロット
fig, ax1 = plt.subplots(figsize=(10, 5))
steps = list(results.keys())
mses = [results[s]["mse"] for s in steps]
times = [results[s]["time"] for s in steps]

ax1.plot(steps, mses, "b-o", linewidth=2, label="MSE")
ax1.set_xlabel("推論ステップ数", fontsize=12)
ax1.set_ylabel("MSE", color="b", fontsize=12)
ax1.tick_params(axis="y", labelcolor="b")

ax2 = ax1.twinx()
ax2.plot(steps, times, "r-s", linewidth=2, label="推論時間")
ax2.set_ylabel("推論時間 (秒)", color="r", fontsize=12)
ax2.tick_params(axis="y", labelcolor="r")

plt.title("推論ステップ数 vs 品質・速度のトレードオフ", fontsize=14)
fig.legend(loc="upper right", bbox_to_anchor=(0.88, 0.88))
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
```

</details>

---
## まとめ

このノートブックでは、**Flow Matching** によるロボット行動生成を学びました。

### 重要なポイント:

1. **Flow Matching の原理**: ノイズからデータへの直線的な経路（線形補間）を学習する
2. **シンプルな損失関数**: $\mathcal{L} = \mathbb{E}[\|v_\theta(x_t, t) - (x_1 - x_0)\|^2]$
3. **Euler 積分によるサンプリング**: 少ないステップ（10-20）で高品質な行動生成が可能
4. **Diffusion との比較**: 数学的にシンプルで、学習が安定し、推論が高速

### 次のステップ:
- 最適輸送 (Optimal Transport) によるペアリング
- より高度な ODE ソルバー（Heun 法、RK4）の利用
- 実ロボットタスクへの適用